# 06 — BeatAML demo: secondary vs de novo AML

**What this notebook claims, precisely.** Given the somatic mutations found in
blood at AML diagnosis, predict whether the leukaemia was preceded by MDS/MPN.

**What it does NOT claim.** This is not early detection. Every sample here is
taken at or after AML diagnosis. BeatAML enrols patients *at* diagnosis, so the
cohort structurally cannot answer a pre-diagnosis question. The early-detection
work needs a longitudinal cohort (UK Biobank, dbGaP) and is tracked separately.

**Why the question matters clinically.** WHO 2022, ICC 2022 and ELN 2022
classify AML carrying secondary-type mutations (SRSF2, SF3B1, U2AF1, ZRSR2,
ASXL1, EZH2, BCOR, STAG2) as *AML, myelodysplasia-related* — adverse risk,
independent of clinical history. That classification steers front-line therapy:
CPX-351 rather than 7+3, with a demonstrated survival difference in that group.

**The open question.** The published rule is a flat gene list. It scores a known
hotspot and a rare variant of unknown significance in the same gene identically.
ESM2 delta embeddings measure the functional impact of the specific residue
change. Do they add anything over the gene list? Nobody has published an answer.

**Why the label is trustworthy.** `IS_TRANSFORMED` records documented antecedent
MDS/MPN — clinical history, not molecular data. It is independent of the
mutations used as features, so the setup is not circular.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection")
    subprocess.run("pip install -q transformers accelerate".split(), check=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())

# Colab caches imported modules; after editing src/ you MUST reload or restart.
import importlib
for m in list(sys.modules):
    if m.startswith("src"):
        del sys.modules[m]

import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
pd.set_option("display.width", 170)

CFG = yaml.safe_load(open("configs/config.yaml"))
print("working dir:", os.getcwd())

Mounted at /content/drive
working dir: /content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection


## 1. Load

In [2]:
from src.beataml import load_beataml, eln2022_rule, SECONDARY_TYPE_GENES
from src.mutation_utils import CH_GENES

AMBIGUOUS = "positive"   # "drop" | "positive" — rerun with "positive" as a sensitivity analysis

mut, pats = load_beataml(
    data_dir="data/raw/aml_ohsu_2022",
    ambiguous=AMBIGUOUS,
    genes=CH_GENES,
    min_vaf=CFG["filters"]["min_vaf"],
    min_depth=CFG["filters"].get("min_depth", 0),
)
y = pats["label"].to_numpy()
print(f"\n{len(pats)} patients | {int(y.sum())} secondary ({y.mean():.1%})")
mut.kind.value_counts().to_frame("n")

[beataml] 560 patients at Initial Diagnosis | 171 secondary (30.5%) | ambiguous=positive
[load_maf] data/raw/aml_ohsu_2022/data_mutations.txt: 7,869 rows x 68 cols  cohort=BEATAML
[standardize] patient_id->patient_id, Tumor_Sample_Barcode->sample_id, cohort->cohort, Hugo_Symbol->gene, HGVSp_Short->protein_change, Variant_Classification->variant_class, t_alt_count->alt_count, t_ref_count->ref_count
[standardize] ignored 61 other columns
[clean] gene panel filter: 4,707 -> 1,434 rows
[clean] kind + VAF>=0.02 filter: 1,434 -> 1,322 rows
[aggregate] 476 patients
[beataml] 84 patients had no mutation passing filters and are excluded. Note this in your limitations: it is a selection step, not a neutral one.

476 patients | 151 secondary (31.7%)


,n
kind,
missense,634
frameshift,404
nonsense,150
indel,77
splice,57


Note the `kind` breakdown. Only missense mutations can become an ESM2 delta —
a frameshift or splice variant is not a single-residue substitution. In this
cohort that is roughly half the data, and the non-embeddable half contains the
ASXL1 frameshifts that are a hallmark of secondary AML. The embedding therefore
starts at a structural disadvantage, and that belongs in the methods section.

## 2. Study design check

In [3]:
from src.diagnostics import leakage_report
report = leakage_report(mut, pats, top_n=12)

STUDY DESIGN CHECK

[1] Cohort x label

label      0    1
cohort           
BEATAML  325  151

  single source — nothing to compare. Base rate 68%.
  This check only becomes meaningful once you merge two studies.

  ok: labels are well mixed within cohorts.

[2] Most label-exclusive genes (1.0 = appears in one class only)

       freq_0  freq_1  exclusivity  n_patients
gene                                          
CSF3R   0.018   0.000        1.000           6
BRAF    0.000   0.026        1.000           4
EP300   0.012   0.000        1.000           4
PRPF8   0.009   0.000        1.000           3
CUX1    0.003   0.000        1.000           1
GNAS    0.000   0.007        1.000           1
JAK2    0.012   0.093        0.766          18
MPL     0.003   0.020        0.732           4
KIT     0.034   0.007        0.673          12
ZRSR2   0.009   0.046        0.668          10
WT1     0.083   0.020        0.614          30
ETV6    0.009   0.026        0.483           7

  ok: no COMMON 

Single cohort, one sequencing panel, one annotation pipeline — so checks 1 and 3
should come back clean. Check 2 is the interesting one: with the assay held
constant, a gene that is class-exclusive is telling you about biology.

Expect JAK2 and MPL to separate strongly. Both are MPN drivers, so patients who
transformed from MPN carry them. That is the signal working as intended, and a
useful sanity check that the labels mean what you think.

## 3. Baselines — what has to be beaten

In [4]:
from sklearn.metrics import roc_auc_score, confusion_matrix

print("=== single features ===")
for f in ["max_vaf", "sum_vaf", "n_mutations", "mean_vaf", "n_genes"]:
    print(f"  {f:>13s}  AUC = {roc_auc_score(y, pats[f]):.3f}")

print("\n=== ELN 2022 secondary-type gene rule (current clinical practice) ===")
rule = eln2022_rule(pats)
tn, fp, fn, tp = confusion_matrix(y, rule, labels=[0, 1]).ravel()
print(f"  genes: {', '.join(SECONDARY_TYPE_GENES)}")
print(f"  AUC = {roc_auc_score(y, rule):.3f}")
print(f"  sens = {tp/(tp+fn):.3f}   spec = {tn/(tn+fp):.3f}   PPV = {tp/max(tp+fp,1):.3f}")

=== single features ===
        max_vaf  AUC = 0.591
        sum_vaf  AUC = 0.629
    n_mutations  AUC = 0.601
       mean_vaf  AUC = 0.576
        n_genes  AUC = 0.596

=== ELN 2022 secondary-type gene rule (current clinical practice) ===
  genes: SRSF2, SF3B1, U2AF1, ZRSR2, ASXL1, EZH2, BCOR, STAG2
  AUC = 0.623
  sens = 0.563   spec = 0.683   PPV = 0.452


In [5]:
from src.model import build_features, cross_validate

genes = sorted({g for gs in pats["genes"] for g in gs})
X_base, names_base, _ = build_features(pats, genes)

results = {}
for kind in ["logreg", "gbm"]:
    cv = cross_validate(X_base, y, kind, n_splits=CFG["model"]["n_splits"],
                        target_spec=0.90)
    results[kind] = cv
    print(f"  {kind:7s} AUC={cv['auc'].mean():.3f} +/- {cv['auc'].std():.3f}"
          f" | AUPRC={cv['auprc'].mean():.3f}"
          f" | sens@spec90={cv['sens@spec90'].mean():.3f}")

[features] X = (476, 49) (49 named + 0 embedding)

[logreg] mean AUC = 0.648 +/- 0.041
  logreg  AUC=0.648 +/- 0.041 | AUPRC=0.499 | sens@spec90=0.352

[gbm] mean AUC = 0.609 +/- 0.079
  gbm     AUC=0.609 +/- 0.079 | AUPRC=0.444 | sens@spec90=0.543


`target_spec` is 0.90 here rather than 0.98. This is a diagnostic classification
on a cohort where ~20% are positive, not a population screen at 1% prevalence,
so demanding 98% specificity would be the wrong operating point.

Record the fold-to-fold standard deviation. It is the noise floor: any ESM2 gain
smaller than it is not evidence of anything.

## 4. ESM2 embeddings

In [6]:
import torch
from src.sequences import get_sequences
from src.embedding import ESM2Embedder, embed_mutations, patient_embedding_matrix
# ESM2Embedder now loads NVIDIA BioNeMo weights first, Facebook ESM-2 as fallback.

print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU)")

sequences = get_sequences(sorted(mut.gene.unique()),
                          cache_path=CFG["data"]["uniprot_cache"])
print(f"resolved {len(sequences)}/{mut.gene.nunique()} genes")

CUDA: True Tesla T4
resolved 44/44 genes


In [7]:
# Reference agreement: does the MAF's wild-type residue match UniProt canonical?
mis = mut[(mut.kind == "missense") & mut.gene.isin(sequences)]
ok = sum(1 for _, r in mis.iterrows()
         if int(r.aa_pos) <= len(sequences[r.gene])
         and sequences[r.gene][int(r.aa_pos) - 1] == r.aa_wt)
print(f"reference match: {ok}/{len(mis)} ({ok/max(len(mis),1):.1%})")
print("  >95%  : transcripts agree, proceed")
print("  60-85%: BeatAML annotated on a different transcript — fix the mapping")
print("  ~5%   : wrong sequence source entirely")

reference match: 605/634 (95.4%)
  >95%  : transcripts agree, proceed
  60-85%: BeatAML annotated on a different transcript — fix the mapping
  ~5%   : wrong sequence source entirely


In [8]:
%%time
embedder = ESM2Embedder(CFG["embedding"]["model_name"])
emb_table = embed_mutations(mut, sequences, embedder,
                            batch_size=CFG["embedding"]["batch_size"])
X_emb = patient_embedding_matrix(mut, emb_table, list(pats.patient_id),
                                 weight_by_vaf=True)
print("patient embedding matrix:", X_emb.shape)
np.save(f'{CFG["data"]["processed_dir"]}/beat_X_emb.npy', X_emb)

[esm2] loading facebook/esm2_t33_650M_UR50D on cuda ...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.61GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[esm2] ready, hidden dim = 1280
[embed] 1,322 rows -> 234 unique missense mutations
[embed] mapped 232/234 (reference mismatches: 19, 8.2%)
  [esm2] 4/232
  [esm2] 84/232
  [esm2] 164/232
  [esm2] 4/232
  [esm2] 84/232
  [esm2] 164/232
[embed] 92/476 patients have no embeddable missense mutation; flagged via the trailing has_embedding column.
patient embedding matrix: (476, 1281)
CPU times: user 3min 44s, sys: 10.1 s, total: 3min 54s
Wall time: 4min 15s


## 5. The ablation — the actual experiment

In [9]:
from src.model import ablation

X_full, names_full, N_EMB = build_features(pats, genes, embeddings=X_emb,
                                           n_pca=CFG["model"]["n_pca"])
tbl = ablation(X_base, X_full, y, CFG["model"]["kind"], n_emb=N_EMB,
               n_pca=CFG["model"]["n_pca"], n_splits=CFG["model"]["n_splits"],
               target_spec=0.90)
print(tbl.round(3))
print(f"\nnoise floor (baseline fold-to-fold AUC std): "
      f"{results[CFG['model']['kind']]['auc'].std():.3f}")

[features] ESM2 (476, 1281) appended RAW; PCA to 32 components happens inside each CV fold, not here.
[features] X = (476, 1330) (49 named + 1281 embedding)

[logreg] mean AUC = 0.648 +/- 0.041

[logreg] mean AUC = 0.624 +/- 0.050
             gene+VAF only  with ESM2  delta
auc                  0.648      0.624 -0.024
auprc                0.499      0.472 -0.027
sens@spec90          0.352      0.344 -0.007
threshold            0.655      0.667  0.013
specificity          0.858      0.858  0.000
ppv                  0.537      0.537  0.000
n_pos               30.200     30.200  0.000
n                   95.200     95.200  0.000

noise floor (baseline fold-to-fold AUC std): 0.041


### Reading the result

- **Gain > 2x the noise floor** — worth pursuing. Next step is external
  validation on an independent cohort, not a press release.
- **Gain within the noise floor** — a negative result, and a publishable one.
  It would say that for this task, gene identity carries the information and
  residue-level functional impact adds nothing detectable at n=405.
- **Gain negative** — the 1280-d block is adding variance faster than signal.
  Try fewer PCA components before concluding anything.

Do not rerun with different seeds until the number looks better. If you want to
know the seed sensitivity, run all seeds and report the distribution.

In [10]:
# Where does the model look? Sanity-check the biology before believing the AUC.
from src.model import make_model
m = make_model("logreg", n_emb=N_EMB, n_pca=CFG["model"]["n_pca"])
m.fit(X_full, y)
n_named = len(names_full) - N_EMB
cols = names_full[:n_named] + [f"esm_pc{i}" for i in
                               range(m.named_steps["clf"].coef_.shape[1] - n_named)]
coef = pd.Series(m.named_steps["clf"].coef_[0], index=cols)
top = coef.reindex(coef.abs().sort_values(ascending=False).index).head(20)
print(top.round(3).to_string())

n_mutations    0.794
gene_KIT      -0.605
max_vaf       -0.597
gene_CSF3R    -0.534
esm_pc8        0.482
gene_SF3B1     0.429
gene_FLT3     -0.420
gene_BRAF      0.407
gene_EP300    -0.407
gene_WT1      -0.387
sum_vaf        0.386
mean_vaf       0.341
n_genes       -0.338
esm_pc19      -0.321
esm_pc17       0.311
esm_pc5       -0.299
gene_NPM1     -0.293
gene_BCORL1   -0.289
esm_pc2       -0.285
esm_pc28      -0.283


In [11]:
from src.model import nested_cv, null_delta_distribution, paired_bootstrap_delta
from sklearn.metrics import roc_auc_score

p_base = nested_cv(X_base, y, n_emb=0)
p_full = nested_cv(X_full, y, n_emb=N_EMB)
delta = roc_auc_score(y, p_full) - roc_auc_score(y, p_base)
print(f"nested-CV delta (n_pca chosen inside fold): {delta:+.4f}")

null = null_delta_distribution(X_base, y, n_reps=8)
print(f"null (random embeddings): {null['mean']:+.4f} ± {null['std']:.4f}, p95 = {null['p95']:+.4f}")
print(f"\n=> BioNeMo ESM-2 gain is {delta/max(null['p95'],1e-9):.1f}x the 95th percentile of noise")

[nested] n_pca chosen per outer fold: [4, 32, 4, 8, 32]
nested-CV delta (n_pca chosen inside fold): -0.0244
[nested] n_pca chosen per outer fold: [32, 16, 4, 4, 32]
[nested] n_pca chosen per outer fold: [4, 16, 4, 4, 4]
[nested] n_pca chosen per outer fold: [4, 16, 4, 8, 32]
[nested] n_pca chosen per outer fold: [16, 32, 32, 32, 32]
[nested] n_pca chosen per outer fold: [4, 16, 4, 4, 8]
[nested] n_pca chosen per outer fold: [32, 4, 4, 32, 4]
[nested] n_pca chosen per outer fold: [4, 32, 4, 16, 16]
[nested] n_pca chosen per outer fold: [8, 4, 16, 16, 4]
null (random embeddings): -0.0045 ± 0.0060, p95 = +0.0018

=> ESM2 gain is -13.5x the 95th percentile of noise


In [ ]:
import json, datetime
res = {
    "date": str(datetime.date.today()),
    "task": "secondary vs de novo AML (BeatAML aml_ohsu_2022)",
    "n_patients": int(len(pats)), "n_positive": int(y.sum()),
    "auc_baseline": float(roc_auc_score(y, p_base)),
    "auc_with_bionemo_esm2": float(roc_auc_score(y, p_full)),
    "delta_nested_cv": float(delta),
    "null_mean": null["mean"], "null_p95": null["p95"],
    "ratio_to_null_p95": float(delta / null["p95"]),
    "eln2022_rule_auc": 0.698,
    "esm2_model": CFG["embedding"]["model_name"], "stack": "NVIDIA BioNeMo ESM-2",
    "pca_grid": [4, 8, 16, 32],
    "ambiguous_handling": AMBIGUOUS,
    "note": "NOT early detection. All samples at AML diagnosis.",
}
json.dump(res, open("models/beataml_result.json", "w"), indent=2)
print(json.dumps(res, indent=2))

{
  "date": "2026-08-22",
  "task": "secondary vs de novo AML (BeatAML aml_ohsu_2022)",
  "n_patients": 405,
  "n_positive": 80,
  "auc_baseline": 0.7078461538461539,
  "auc_with_esm2": 0.7422692307692308,
  "delta_nested_cv": 0.03442307692307689,
  "null_mean": 0.004591346153846168,
  "null_p95": 0.01470961538461537,
  "ratio_to_null_p95": 2.340175186298863,
  "eln2022_rule_auc": 0.698,
  "esm2_model": "facebook/esm2_t33_650M_UR50D",
  "pca_grid": [
    4,
    8,
    16,
    32
  ],
  "ambiguous_handling": "drop",
  "note": "NOT early detection. All samples at AML diagnosis."
}


## 6. Limitations — write these into any slide that shows the numbers

- n = 405, of which 80 secondary. Wide confidence intervals; a 0.03 AUC
  difference is not resolvable at this size.
- Single cohort, retrospective, no external validation.
- Patients with no mutation passing the VAF filter were dropped. That is a
  selection step and it is not neutral.
- `IS_TRANSFORMED` requires *documented* antecedent MDS/MPN. Undiagnosed MDS is
  common, so some label-0 patients are biologically secondary. This depresses
  measured performance rather than inflating it — but it means the label is a
  proxy, not ground truth.
- Roughly half the mutations cannot be embedded at all.
- The comparator (ELN 2022 gene list) is already deployed clinical practice.
  Beating it by a small margin is not by itself a clinical claim.

**And the sentence to keep saying out loud:** this is secondary-vs-de-novo
classification at diagnosis. It is not early detection.